# GroundGate — quickstart notebook

Claim-by-claim grounding verification for RAG answers, at runtime.

This notebook uses only the deterministic core — no API key, no LLM, no network.

In [ ]:
# %pip install groundgate  # or, from a repo checkout: %pip install -e ..
from groundgate import verify

## 1. A faithful answer passes

Context is what your retriever returned; the answer cites it with `[id]` markers.

In [ ]:
context = [
    {
        "id": "policy",
        "text": (
            "O plano Essencial cobre consultas de emergencia com carencia de "
            "24 horas. O limite anual de reembolso e de R$ 50.000."
        ),
    }
]

faithful = (
    "A carencia para emergencias e de 24 horas [policy]. "
    "O limite anual de reembolso e de R$ 50.000 [policy]."
)

verdict = verify(faithful, context)
verdict.status, verdict.score

## 2. A hallucinated number fails — with the exact claim and reason

In [ ]:
unfaithful = (
    "A carencia para emergencias e de 12 horas [policy]. "
    "O limite anual de reembolso e de R$ 50.000 [policy]."
)

verdict = verify(unfaithful, context)
for claim in verdict.claims:
    print("PASS" if claim.supported else "FAIL", "-", claim.claim.text)
    print("      ", claim.reason)

## 3. The structured verdict

Everything is serializable — this is the same payload the MCP server and CLI return.

In [ ]:
import json

print(json.dumps(verdict.to_dict(), ensure_ascii=False, indent=2))

## 4. Optional: an LLM judge for ambiguous paraphrases

The deterministic core decides most claims. Only *ambiguous* ones (partial
token overlap) escalate to a judge — pick any provider:

```python
from groundgate import GroundGate
from groundgate.providers import AnthropicJudge  # or OpenAIJudge, OllamaJudge, BedrockJudge

gate = GroundGate(judge=AnthropicJudge())  # needs pip install groundgate[anthropic]
verdict = gate.verify(answer, context)
```

With no judge configured the gate **fails closed**: ambiguous claims are
rejected, never silently accepted.